# π0.5 LIBERO Replication — Colab (A100)

One unattended run: **baseline eval → train 30k steps → evaluate every 5k checkpoint**.

**Storage plan**
| Artifact | Where | Notes |
|---|---|---|
| Full checkpoints (6 × 4.8 GB) | Colab local disk `/content` | ephemeral, only for resume + eval |
| LoRA adapters (6 × 84 MB) | Google Drive | permanent, the real output |
| Eval metrics + videos | WandB | permanent |
| Norm stats | git (already committed) | cloned with the repo |

**How to use:** set runtime to A100 (Runtime → Change runtime type), fill in §0, then run §1 then §2. That's it — §2 runs the whole experiment with no further intervention.

---
## §0 · Config

In [ ]:
# ── Tokens ────────────────────────────────────────────────────────────────────
WANDB_API_KEY = ""          # from wandb.ai/authorize
HF_TOKEN      = ""          # from huggingface.co/settings/tokens (read is enough)

# ── Sources ───────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/LavetteSinsora/pi05-libero-replication"
HF_DATASET  = "pi05-libero/libero_object_summed_subsampling"
BASE_CKPT   = "gs://openpi-assets/checkpoints/pi05_base"

# ── Fixed names (match the TrainConfig) ────────────────────────────────────────
CONFIG_NAME = "pi05_libero_object_lora"
EXP_NAME    = "masked_loss_summed_subsampling"

# ── Paths ────────────────────────────────────────────────────────────────────
REPO         = "/content/pi05-libero-replication"
DATASET_DIR  = "/content/lerobot/libero_object_summed_subsampling"
LOCAL_CKPT   = "/content/checkpoints/pi05_libero"          # full checkpoints (ephemeral)
DRIVE_ROOT   = "/content/drive/MyDrive/pi05_libero_replication"
DRIVE_LORA   = f"{DRIVE_ROOT}/lora/{EXP_NAME}"             # permanent LoRA adapters

# ── Knobs (leave as-is for the full experiment) ─────────────────────────────────
RUN_BASELINE = True
TRIALS_PER_TASK = 50       # 10 tasks × 50 = 500 rollouts per eval

---
## §1 · Setup

In [ ]:
# 1a. Mount Drive + clone repo
import os, pathlib, subprocess
from google.colab import drive

drive.mount("/content/drive")

if not pathlib.Path(REPO).exists():
    subprocess.run(["git", "clone", "--recurse-submodules", GITHUB_REPO, REPO], check=True)
print("Repo ready.")

In [ ]:
# 1b. System libs + Python packages
subprocess.run(["apt-get", "install", "-y", "-qq",
                "ffmpeg", "libgl1-mesa-glx", "libegl1-mesa"], check=True)

!pip install -q -e {REPO}/third_party/openpi
!pip install -q -e {REPO}/third_party/libero
print("Packages installed.")

In [ ]:
# 1c. Environment variables
os.environ["HF_LEROBOT_HOME"]                = "/content/lerobot"
os.environ["MUJOCO_GL"]                       = "egl"     # GPU off-screen rendering
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]  = "0.9"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

for d in [LOCAL_CKPT, DRIVE_LORA]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("Env set.")

In [ ]:
# 1d. Download dataset from HF Hub to local disk (~315 MB, ~1-2 min)
from huggingface_hub import snapshot_download

if pathlib.Path(DATASET_DIR).exists() and any(pathlib.Path(DATASET_DIR).iterdir()):
    print("Dataset already present.")
else:
    snapshot_download(repo_id=HF_DATASET, repo_type="dataset", local_dir=DATASET_DIR)
    print("Dataset downloaded.")

In [ ]:
# 1e. GCS auth (to read the public pi05_base checkpoint) + norm-stats sanity check
from google.colab import auth
auth.authenticate_user()

norm_stats = pathlib.Path(
    f"{REPO}/assets/pi05_libero/{CONFIG_NAME}/libero_object_summed_subsampling/norm_stats.json"
)
assert norm_stats.exists(), (
    f"norm_stats.json missing at {norm_stats} — it must be committed to the repo. "
    "Run compute_norm_stats.py locally and commit assets/ before training."
)
print("Norm stats present. Setup complete.")

---
## §2 · Run the full experiment (unattended)

This single cell:
1. **Baseline eval** — pretrained π0.5 on LIBERO-OBJECT (our norm stats) → WandB run `pi05_base_benchmark`.
2. **Train 30k steps** — one continuous run. `keep_period=5000` preserves checkpoints at 5000/10000/15000/20000/25000 + final. Loss curve → WandB.
3. **Eval every checkpoint** — each preserved checkpoint → WandB run `step_<n>` (metrics + videos), and its LoRA adapters (84 MB) → Drive.

Everything runs as subprocesses so each stage gets a clean GPU. Estimated total: ~6-9 h on A100.

Progress is visible live in WandB under project `pi05_libero_replication`.

In [ ]:
import subprocess, pathlib, sys

OPENPI = f"{REPO}/third_party/openpi"
CKPT_RUN = pathlib.Path(f"{LOCAL_CKPT}/{CONFIG_NAME}/{EXP_NAME}")

def sh(cmd, cwd=None):
    """Run a command, streaming output; raise on failure."""
    print(f"\n$ {' '.join(cmd)}\n", flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)

def benchmark(checkpoint_dir, exp_dir, train_step=None):
    cmd = [
        sys.executable, f"{REPO}/scripts/benchmark.py",
        "--config_name", CONFIG_NAME,
        "--checkpoint_dir", str(checkpoint_dir),
        "--exp_dir", str(exp_dir),
        "--num_trials_per_task", str(TRIALS_PER_TASK),
    ]
    if train_step is not None:
        cmd += ["--train_step", str(train_step)]
    sh(cmd, cwd=OPENPI)   # cwd=openpi so ../../assets path resolves

# ── 1. Baseline ────────────────────────────────────────────────────────────────
if RUN_BASELINE:
    print("=" * 70 + "\nBASELINE EVALUATION\n" + "=" * 70)
    benchmark(BASE_CKPT, f"/content/experiments/pi05_base_benchmark")

# ── 2. Train 30k ──────────────────────────────────────────────────────────────
print("=" * 70 + "\nTRAINING (30k steps)\n" + "=" * 70)
sh([
    sys.executable, "scripts/train.py", CONFIG_NAME,
    "--exp_name", EXP_NAME,
    "--checkpoint_base_dir", LOCAL_CKPT,
], cwd=OPENPI)

# ── 3. Evaluate every preserved checkpoint + save LoRA to Drive ────────────────
steps = sorted(int(d.name) for d in CKPT_RUN.iterdir() if d.is_dir() and d.name.isdigit())
print("=" * 70 + f"\nEVALUATING CHECKPOINTS: {steps}\n" + "=" * 70)

for step in steps:
    ckpt = CKPT_RUN / str(step)
    benchmark(ckpt, f"/content/experiments/{CONFIG_NAME}/{EXP_NAME}/step_{step}", train_step=step)
    sh([
        sys.executable, f"{REPO}/scripts/extract_lora.py",
        "--checkpoint_dir", str(ckpt),
        "--out", f"{DRIVE_LORA}/step_{step}.npz",
    ])

print("\n" + "=" * 70 + "\nDONE. Metrics + videos in WandB; LoRA adapters on Drive.\n" + "=" * 70)

---
## §3 · (Optional) Results summary

All numbers are already in WandB. This just prints a local table from the saved `results.json` files.

In [ ]:
import json, pathlib

def rate(p):
    p = pathlib.Path(p)
    return json.loads(p.read_text())["aggregate_success_rate"] if p.exists() else None

rows = []
b = rate("/content/experiments/pi05_base_benchmark/results.json")
if b is not None:
    rows.append(("baseline (pi05_base)", b))

run_dir = pathlib.Path(f"/content/experiments/{CONFIG_NAME}/{EXP_NAME}")
if run_dir.exists():
    for d in sorted(run_dir.iterdir(), key=lambda x: int(x.name.split('_')[-1]) if x.name.startswith('step_') else -1):
        r = rate(d / "results.json")
        if r is not None:
            rows.append((d.name, r))

print(f"{'run':<28}{'success':>10}")
print("-" * 38)
for name, r in rows:
    print(f"{name:<28}{r:>9.1%}")